<a href="https://colab.research.google.com/github/akwve/STA160_Group2_Project/blob/main/FULL_TWITTER_SET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# --- Mount Drive & resolve a stable symlink /content/model ---
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, pathlib, sys

def find_hf_model_dir(root: str):
    for dirpath, dirnames, filenames in os.walk(root):
        if "config.json" in filenames and any(f.endswith((".bin", ".safetensors")) for f in filenames):
            return dirpath
    return None

# Try both "MyDrive" and "My Drive"
base_candidates = [
    "/content/drive/MyDrive/Bert_Models/bert_fake_news_model",
    "/content/drive/My Drive/Bert_Models/bert_fake_news_model",
]
drive_root = next((p for p in base_candidates if os.path.exists(p)), None)
assert drive_root, "Model folder not found in Drive."

model_dir = find_hf_model_dir(drive_root)
assert model_dir, "Could not find config.json + weights under your Drive model folder."

# Make a stable symlink so your code can always load from /content/model
stable = "/content/model"
if os.path.islink(stable) or os.path.exists(stable):
    os.unlink(stable) if os.path.islink(stable) else shutil.rmtree(stable)
os.symlink(model_dir, stable, target_is_directory=True)

print("Resolved model_dir:", model_dir)
print("Stable path -> /content/model")

Mounted at /content/drive
Resolved model_dir: /content/drive/MyDrive/Bert_Models/bert_fake_news_model/content/bert_fake_news_model
Stable path -> /content/model


In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from openai import OpenAI

# ---- 1. Locate the local HF model directory ----
def find_hf_model_dir(root: str):
    for dirpath, dirnames, filenames in os.walk(root):
        has_config = "config.json" in filenames
        has_weights = any(f.endswith(".bin") or f.endswith(".safetensors") for f in filenames)
        if has_config and has_weights:
            return dirpath
    return None

unzip_root = "/content/bert_fake_news_model"
model_dir = find_hf_model_dir("/content/drive/MyDrive/Bert_Models/bert_fake_news_model")

if not model_dir:
    raise FileNotFoundError(f"Couldn't find a model under {unzip_root}")

print("Using model_dir:", model_dir)
print("Files:", os.listdir(model_dir))

# ---- 2. Load local model ----
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True).to(device)
model.eval()

# ---- 3. Initialize OpenAI client ----
client = OpenAI(api_key="your_openai_api_key_here")  # replace with your OpenAI API key

# ---- 4. Combined classification + ChatGPT analysis loop ----
while True:
    text = input("\nEnter tweet to classify (or 'quit' to exit): ")
    if text.strip().lower() == "quit":
        break

    # Hugging Face prediction
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        pred = probs.argmax(dim=-1).item()

    label = "Fake" if pred == 0 else "Real"
    confidence = probs[0, pred].item()

    # ChatGPT fact-checking
    long_text = f"""Consider the following example: ”INPUT”.
The returned answer should be determining the truth value of the inputted tweet in the following format:
“True, confidence level = 0.97”.
Do not deviate from the stated format. Give a binary answer of “True” or “False” with an accompanying confidence level.
The confidence level should be replicable with the same inputted tweet. Make the confidence level as precise and accurate as possible.
Spend more time calculating if necessary. This is on a 0–1 predictive scale. Do not give a confidence level of less than 0.5.
Determine if the following tweet contains true or false information: {text}
"""

    response = client.chat.completions.create(
        model="gpt-5",
        messages=[{"role": "user", "content": long_text}]
    )

    print(f"\n[Local Model] → Prediction: {label} (confidence: {confidence:.3f})")
    print(f"\n[ChatGPT] → {response.choices[0].message.content}")

Using model_dir: /content/drive/MyDrive/Bert_Models/bert_fake_news_model/content/bert_fake_news_model
Files: ['config.json', 'tokenizer_config.json', 'model.safetensors', 'vocab.txt', 'tokenizer.json', 'special_tokens_map.json']
Using device: cpu

Enter tweet to classify (or 'quit' to exit): the “delete republican districts act” is up by 14 points in the former heart of the CA GOP

[Local Model] → Prediction: Fake (confidence: 0.897)

[ChatGPT] → False, confidence level = 0.76

Enter tweet to classify (or 'quit' to exit): quit
